In [11]:
##### IMPORTS

import os
os.environ['picaso_refdata'] = r'C:\Users\Alex\Desktop\Picaso\picaso\reference' # THIS MUST GO BEFORE YOUR IMPORT STATEMENT
os.environ['PYSYN_CDBS'] = r'C:\Users\Alex\Desktop\Picaso\grp\redcat\trds' # This is for the stellar data discussed below.

# General
import gc
import numpy as np
import astropy.units as u
import bd_support as sup

from pathlib import Path
from itertools import product

# Picaso and Virga
from picaso import justdoit as jdi
from virga import justdoit as vj

# To see what clouds are availible
# vj.available()

In [12]:
##### CONFIGURATIONS

# Directories
sonor_path  = r'C:\Users\Alex\Desktop\Picaso\data\sonora' # Sonora db
# sonor_path  = '/groups/tkaralidi/pbraunschweig/training_set/profiles/'
virga_path  = r'C:\Users\Alex\Desktop\Picaso\data\virga'  # Virga
# virga_path  = '/home/sa221179/picaso/virga/'
# opaci_path  = None # Opacity db
opaci_path  = r'D:\opacity_500k_for_R5000_egpoutput.db'
output_path = Path(r"C:\Users\Alex\Desktop\Picaso\NN_project\outputs")
# output_path = 'home/al864695/ouputs'

# Constant values
wav_range   = [0.5, 30.0] # microns
MH          = 1.0        # [M/H] metallicity factor ~ solar
MU          = 2.36       # Average MU
# R           = 300        # resolution
R           = 5000       # resolution

# Dictionary for cloud naming conventions
cloud_dict  = {'Fe': '1', 'H2O': '2', 'KCl': '3', 'Mg2SiO4': '4'}
               #'MgSiO3': '5', 'MnS': '6', 'NH3': '7', 'Na2S': '8'}

In [13]:
##### METHOD BROWN DWARF SPECTRUM=


def bd_spectrum(Teff, logg, fsed, kzz):
    bd  = jdi.inputs(calculation="browndwarf")
    bd.phase_angle(0)

    gravity = 10**logg * 1e-2  # cgs -> m s^-2
    bd.gravity(gravity, gravity_unit=u.Unit('m/s**2'))
    bd.sonora(sonor_path, Teff)

    # TP correction & Kzz
    sup.inject_corr(bd, Teff, logg, fsed)
    prof = bd.inputs['atmosphere']['profile']
    P    = np.asarray(prof["pressure"], float)
    T    = np.asarray(prof['temperature'], float)
    bd.inputs["atmosphere"]["profile"]["kz"] = [float(kzz)] * len(P)

    # Clouds: keep only your four
    allowed  = {'Fe','H2O','KCl','Mg2SiO4'}   # adjust as needed
    rec      = vj.recommend_gas(P, T, MH, MU, plot=False)
    clouds   = [sp for sp in rec if sp in allowed]
    cl_names = ''.join(sorted(cloud_dict[c] for c in clouds))
    bd.virga(clouds, virga_path, fsed, mh=MH, mmw=MU)

    # Chunked opacity + spectrum
    wl_min, wl_max = wav_range
    NCHUNKS        = 24            # bump this up if memory is still tight
    edges          = np.linspace(wl_min, wl_max, NCHUNKS + 1)

    wn_all = []
    fl_all = []

    for a, b in zip(edges[:-1], edges[1:]):
        # tiny pad so boundaries have overlap; avoids empty bins
        eps = 1e-6
        lo  = float(max(wl_min, a) ) - eps
        hi  = float(min(wl_max, b) ) + eps
        if hi <= lo:
            continue

        # Build opacity for this small window
        opa_chunk = jdi.opannection([lo, hi], opaci_path)

        # Compute spectrum for this window, ask for full_output so we can
        # reliably grab arrays
        out = bd.spectrum(opa_chunk, full_output=True)

        wn_chunk = np.asarray(out["wavenumber"], dtype=float)   # cm^-1
        fl_chunk = np.asarray(out["thermal"],    dtype=float)   # per cm^-1

        # Skip empty/degenerate returns
        if wn_chunk.ndim != 1 or fl_chunk.ndim != 1:
            del opa_chunk, out
            gc.collect()
            continue
        if wn_chunk.size == 0 or fl_chunk.size == 0:
            del opa_chunk, out
            gc.collect()
            continue

        # Accumulate
        wn_all.append(wn_chunk)
        fl_all.append(fl_chunk)

        del opa_chunk, out, wn_chunk, fl_chunk
        gc.collect()

    # Safety check: make sure we actually collected something
    if not wn_all or not fl_all:
        raise RuntimeError("No spectral points gathered; try increasing NCHUNKS or widening eps.")

    # Stitch & sort by wavenumber
    wn = np.concatenate(wn_all)
    fl = np.concatenate(fl_all)
    idx = np.argsort(wn)
    wn, fl = wn[idx], fl[idx]

    # Deduplicate any tiny overlaps (keep first occurrence)
    keep = np.ones_like(wn, dtype=bool)
    keep[1:] = np.abs(np.diff(wn)) > 0
    wn, fl = wn[keep], fl[keep]

    # Final: regrid in wavenumber space to target R (you keep R=5000)
    wn, fl = jdi.mean_regrid(wn, fl, R=R)

    return fl, cl_names

In [14]:
##### GENERATE AND SAVE SPECTRUM (MARGE format, single-case files)

Teff_s = [1351.6243204380194]      # K
grav_s = 1060.001475631001
logg_s = [np.log10(grav_s * 100)]  # log g cgs
fsed_s = [2.0] 
kzz_s  = [1e9]

for Teff_i, logg_i, fsed_i, kzz_i in product(Teff_s, logg_s, fsed_s, kzz_s):

    # Run spectrum
    F_i, names = bd_spectrum(Teff_i, logg_i, fsed_i, kzz_i)

    # Filename encodes the parameters; ML will parse from name
    fname  = (f"T{float(Teff_i)}g{float(logg_i)}f{float(fsed_i)}"
              f"{sup.format_kzz(kzz_i)}c{names}.npy")
    fpath  = output_path / fname

    np.save(fpath, F_i)